# 03 — Fine-tuning an ASR model on the harness corpus, on a Kaggle TPU

This notebook trains **`facebook/w2v-bert-2.0` with a character CTC head** on the harness's
`training` export, selects a checkpoint on `val`, and scores it once on the `gold` export using
the same script-folded WER the harness reports for Scribe, MAI and Gemini, so the fine-tuned
model can be compared directly with the recognisers the harness already uses.

**What it is for right now.** A pilot on about an hour of labels. That is not enough to train a
good model, so treat the pilot as a check that the data path, the TPU graph and the evaluation
are correct end to end before the corpus reaches its ~14 h training target. The code is the same
at either size; only `EPOCHS` needs revisiting (section 1).

### Why this model

| | w2v-BERT 2.0 + CTC (chosen) | Whisper large-v3 / turbo | Qwen3-ASR-1.7B (Nepali fine-tune) |
|---|---|---|---|
| Size | 0.6 B, encoder only | 1.55 B / 0.8 B, encoder-decoder | 2 B, audio LLM |
| On a TPU | fixed shapes, one forward pass per clip; one graph per length bucket | training is fine; evaluation needs autoregressive `generate` under XLA | same problem as Whisper, at 2× the size |
| Output length | none; the head emits one symbol per 20 ms frame | **448-token cap**: the longest label here is already 428 tokens | LLM context, no practical cap |
| Code-switching | character vocabulary built from the labels covers both scripts | forced language token; English-in-Devanagari transliteration is a known failure mode | Nepali is not among its 52 languages; the community fine-tune saw read Devanagari speech only |
| Failure on silence | cannot invent text | can hallucinate on short or quiet clips | same |

The cost of CTC is that it has no internal language model, so English words are spelled from the
acoustics alone. The standard fix, an n-gram LM from the label text decoded with `pyctcdecode`, is
the first thing to add once there is enough text to build one. The table says what each model is
*expected* to do; nothing here has been measured on this corpus yet, and section 11 is where that
starts.

### What this notebook does not claim

- **A gold WER here is not speaker-held-out.** Gold is chosen per clip (D71), so most gold clips
  come from episodes that also feed train (`episode_spans_pots`), and the voice prints in `EDA` found
  the same speaker on both sides of the line under different shows. Section 11 reports the
  spanning clips separately; neither number says how the model does on an unseen voice.
- **Screened rows are fusion output nobody listened to.** Right now they are most of the
  training export. Training on them teaches the model to imitate the fuser. `INCLUDE_SCREENED`
  switches them off; section 3 prints the mix so the choice is made knowingly.
- **The references were seeded by fusion (D74),** which was built from the three recognisers.
  Their WER against those labels in section 12 is therefore probably flattered by anchoring.

### Running it on Kaggle

1. **Build the dataset locally**, from the repository root:
   ```bash
   python scripts/export_dataset.py --kind all --output-root /tmp/ft-export
   D=/tmp/nepanglish-asr
   mkdir -p $D/analytics $D/harness/backend/app/services $D/harness/config
   cp -r /tmp/ft-export/training /tmp/ft-export/gold $D/
   cp /tmp/ft-export/analytics/{analytics.jsonl,manifest.json} $D/analytics/   # optional: baselines
   cp backend/app/__init__.py $D/harness/backend/app/
   cp backend/app/services/{__init__,fold,normalize}.py $D/harness/backend/app/services/
   cp config/normalization.yaml $D/harness/config/
   rm -rf $D/gold/episodes    # training/episodes already holds every episode gold draws from
   ```
   The `harness/` folder carries the harness's own WER code, so this notebook scores with
   exactly the fold (`fold.py` + the D64 table) that `EDA` and the harness use. Without it the
   notebook falls back to plain WER and says so.
   Delete `gold/episodes` only when every gold episode also has train clips; the notebook tells
   you which episode it cannot find if one is missing.
2. **Upload** `/tmp/nepanglish-asr` as a private Kaggle dataset (web UI, or
   `kaggle datasets create -p /tmp/nepanglish-asr --dir-mode zip`).
3. In the notebook: **Add Input** → the dataset; **Settings → Accelerator → TPU v5e-8**;
   **Internet on** (the pretrained model is a 2.3 GB download from the Hugging Face Hub).
4. **Run All.** Set `SMOKE_TEST = True` in section 1 first if you want a ~5-minute run with a
   tiny random model that exercises every cell before you spend quota on the real one.

Artifacts land in `/kaggle/working/w2vbert-nepanglish/`: the best checkpoint (loadable with
`Wav2Vec2BertForCTC.from_pretrained` on any machine), per-clip gold predictions, the training
curve and a `training_summary.json` carrying the export provenance.

## 1. Configuration

Every knob is in this cell. Paths resolve themselves on Kaggle; the `FT_*` environment variables
exist so the same notebook runs unchanged from the repository root on a workstation.

In [ ]:
import os
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()
#: Off Kaggle: the repository, whether Jupyter started in it or in notebooks/ (as in EDA).
REPO_ROOT = Path("..").resolve() if Path("../exports").is_dir() else Path(".").resolve()


def _find_data_root() -> Path:
    """The dataset holding the exports: the first input directory with a `training` export."""
    if os.environ.get("FT_DATA_ROOT"):
        return Path(os.environ["FT_DATA_ROOT"])
    for manifest in sorted(Path("/kaggle/input").glob("**/training/manifest.json")):
        return manifest.parent.parent
    return REPO_ROOT / "exports"


DATA_ROOT = _find_data_root()
#: Where `backend/app/services/fold.py` and `config/normalization.yaml` live: the `harness/`
#: folder of the uploaded dataset (see the recipe above), or else the repository itself.
HARNESS_SRC = Path(
    os.environ.get(
        "FT_HARNESS_SRC", DATA_ROOT / "harness" if (DATA_ROOT / "harness").is_dir() else REPO_ROOT
    )
)
OUTPUT_DIR = Path(
    os.environ.get(
        "FT_OUTPUT_DIR",
        "/kaggle/working/w2vbert-nepanglish" if ON_KAGGLE else REPO_ROOT / "exports" / "finetune",
    )
)
#: Persistent XLA compilation cache: a re-run in the same session skips recompiling the graphs.
XLA_CACHE_DIR = Path(os.environ.get("FT_XLA_CACHE", "/tmp/xla-cache"))

#: A tiny randomly initialised model and two epochs: checks every cell in minutes, learns nothing.
SMOKE_TEST = os.environ.get("FT_SMOKE_TEST", "0") == "1"

MODEL_ID = "facebook/w2v-bert-2.0"

# --- data -----------------------------------------------------------------------------------
#: Screened rows were accepted without listening (D63). Section 3 prints how many there are.
INCLUDE_SCREENED = True
#: Below this many `val` clips, carve a validation slice out of train instead (section 4).
VAL_MIN_CLIPS = 30
VAL_FALLBACK_FRACTION = 0.08
#: Ingest cuts clips to 2-20 s; anything outside this band is dropped and counted.
MIN_SECONDS, MAX_SECONDS = 1.0, 20.5

# --- batching -------------------------------------------------------------------------------
#: Length buckets. Every bucket is one compiled graph for training and one for evaluation, so
#: fewer buckets compile faster and waste more compute on padding (section 7 prints both).
BUCKET_SECONDS = (7.0, 14.0, 20.5)
#: Audio per optimizer step, summed over the batch. Batch size per bucket follows from it.
BATCH_AUDIO_SECONDS = 320

# --- optimisation ---------------------------------------------------------------------------
#: ~30 suits the one-hour pilot (~20 steps an epoch). At ~14 h an epoch is ~10x longer, and
#: 10-15 epochs is the usual range for CTC fine-tuning at that size.
EPOCHS = 30
PEAK_LR = 5e-5
WARMUP_FRACTION = 0.1
WEIGHT_DECAY = 0.0
MAX_GRAD_NORM = 1.0
#: Stop cleanly before Kaggle ends the session. A TPU session is capped at a few hours; check
#: the limit shown in your notebook settings and leave room for the gold evaluation.
MAX_TRAIN_HOURS = 7.0
#: SpecAugment on the input features, applied on the host (section 7 says why not in the model).
SPEC_AUGMENT = {"freq_masks": 2, "freq_width": 15, "time_mask_rate": 0.05, "time_width": 10}

# --- TPU ------------------------------------------------------------------------------------
#: "dp": every chip holds the whole model (0.6 B params + AdamW is ~9.3 GB of a v5e chip's 16 GB).
#: "fsdp": parameters and optimizer state sharded over the chips. Switch to it on out-of-memory.
PARALLELISM = "dp"
GRADIENT_CHECKPOINTING = True

LOG_EVERY_STEPS = 10
SEED = 20260911
#: e.g. "your-name/w2v-bert-nepanglish"; needs an HF_TOKEN Kaggle secret. None = don't push.
PUSH_TO_HUB_REPO = None

if SMOKE_TEST:
    EPOCHS, PEAK_LR, BATCH_AUDIO_SECONDS, LOG_EVERY_STEPS = 2, 1e-3, 160, 5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"data     {DATA_ROOT.resolve()}")
print(f"harness  {HARNESS_SRC.resolve()}")
print(f"output   {OUTPUT_DIR.resolve()}")
print(f"mode     {'SMOKE TEST (tiny random model)' if SMOKE_TEST else MODEL_ID}")

## 2. Environment and accelerator

Tested against **torch 2.8.0 + torch_xla 2.8.0 + transformers 4.57.1**, running the TPU code path
on XLA:CPU with 8 virtual devices. That test established what the code below depends on:

- **`aten::_ctc_loss` has no XLA lowering** ([pytorch/xla#2399](https://github.com/pytorch/xla/issues/2399)).
  Left alone, it falls back to the CPU at an arbitrary point in the graph. Here the loss is
  computed on the host *deliberately*: the forward graph runs, the logits (a few MB) come back,
  and the gradient goes back through the same transfer. That is two device executions per step,
  the minimum.
- **`aten::glu_backward` has no lowering either.** The conformer's convolution module uses
  `nn.GLU`, which cost one host round trip per layer per step; section 8 swaps it for the
  identical `a * sigmoid(b)`.
- Shapes are static per length bucket, so after the first pass through each bucket nothing
  recompiles. The training log prints the compile count so you can confirm that on the TPU.

If the image's `torch_xla` is older than 2.5, this cell installs the tested pair. If it installs
anything, **restart the kernel once** and run all again.

In [ ]:
import importlib.metadata as md
import subprocess
import sys

TESTED = {"torch": "2.8.0", "torch_xla": "2.8.0", "transformers": "4.57.1"}


def _version(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return None


def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)


def _older_than(version, floor):
    return tuple(int(p) for p in version.split("+")[0].split(".")[:2]) < floor


#: v4 and earlier expose /dev/accel*, v5e exposes /dev/vfio/*.
TPU_PRESENT = (
    any(Path("/dev").glob("accel*"))
    or any(Path("/dev/vfio").glob("[0-9]*"))
    or os.environ.get("PJRT_DEVICE") == "TPU"
)
installed = []  # module names
if TPU_PRESENT and (_version("torch_xla") is None or _older_than(_version("torch_xla"), (2, 5))):
    # The CPU build of torch: torch_xla brings the TPU runtime, and the CUDA wheel is 2 GB of
    # libraries a TPU VM cannot use.
    _pip(f"torch=={TESTED['torch']}", "--index-url", "https://download.pytorch.org/whl/cpu")
    _pip(f"torch_xla[tpu]=={TESTED['torch_xla']}")
    installed += ["torch", "torch_xla"]
if _version("transformers") != TESTED["transformers"]:
    _pip(f"transformers=={TESTED['transformers']}")
    installed.append("transformers")
for package, module in (("soundfile", "soundfile"), ("rapidfuzz", "rapidfuzz"), ("PyYAML", "yaml")):
    if _version(package) is None:
        _pip(package)
        installed.append(module)

stale = [module for module in installed if module in sys.modules]
if stale:
    raise SystemExit(f"reinstalled {stale} after they were imported: restart the kernel, run all")
print("installed:", installed or "nothing")

In [ ]:
import concurrent.futures as cf
import functools
import hashlib
import json
import math
import random
import re
import time
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn.functional as F
import transformers
from rapidfuzz.distance import Levenshtein
from transformers import (
    SeamlessM4TFeatureExtractor,
    Wav2Vec2BertConfig,
    Wav2Vec2BertForCTC,
    Wav2Vec2BertProcessor,
    Wav2Vec2CTCTokenizer,
)
from transformers.models.wav2vec2_bert.modeling_wav2vec2_bert import Wav2Vec2BertEncoderLayer

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 90)
transformers.logging.set_verbosity_error()

USE_XLA = _version("torch_xla") is not None and os.environ.get("FT_DISABLE_XLA") != "1"

if USE_XLA:
    import logging

    # torch_xla logs this on every step when jax is absent; nothing here uses Pallas kernels.
    logging.getLogger().addFilter(lambda record: "jax/pallas" not in record.getMessage())

    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.debug.metrics as xla_metrics
    import torch_xla.distributed.spmd as xs
    import torch_xla.runtime as xr

    try:
        xr.initialize_cache(str(XLA_CACHE_DIR), readonly=False)
    except Exception as exc:  # a cache is an optimisation; never fail the run over it
        print(f"xla cache unavailable: {exc}")
    # SPMD: one Python process drives every chip and the compiler partitions the graph, so there
    # is no multiprocessing to break in a notebook, and the host holds one copy of the data.
    xr.use_spmd()
    N_DEVICES = xr.global_runtime_device_count()
    DEVICE = torch_xla.device() if hasattr(torch_xla, "device") else xm.xla_device()
    #: One mesh axis over every chip. It is named `fsdp` because FSDPv2 requires that name; in
    #: "dp" mode it is simply the batch axis.
    MESH = xs.Mesh(np.arange(N_DEVICES), (N_DEVICES,), ("fsdp",))
    xs.set_global_mesh(MESH)
    sync = torch_xla.sync if hasattr(torch_xla, "sync") else xm.mark_step
    ACCELERATOR = f"{xr.device_type()} x{N_DEVICES} (SPMD)"

    def compile_count():
        """Graphs compiled so far. It must stop rising once every bucket has been seen."""
        return xla_metrics.counter_value("UncachedCompile") or 0
else:
    N_DEVICES = 1
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    MESH = None
    ACCELERATOR = str(DEVICE)

    def sync():
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

    def compile_count():
        return 0


#: bf16 autocast on TPU (and on GPUs that support it); full precision on CPU.
AUTOCAST_DTYPE = (
    torch.bfloat16
    if USE_XLA or (DEVICE.type == "cuda" and torch.cuda.is_bf16_supported())
    else None
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if USE_XLA:
    xm.set_rng_state(SEED)

VERSIONS = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "torch_xla": _version("torch_xla"),
    "transformers": transformers.__version__,
}
print(f"accelerator  {ACCELERATOR}   autocast {AUTOCAST_DTYPE}")
print("versions    ", VERSIONS)
if not USE_XLA and DEVICE.type == "cpu" and not SMOKE_TEST:
    raise SystemExit(
        "no TPU or GPU found, and a full run on a CPU will not finish. On Kaggle: Settings -> "
        "Accelerator -> TPU v5e-8. To exercise the notebook here, set SMOKE_TEST = True."
    )

## 3. The exports: provenance, integrity and composition

Each export directory carries a manifest with a SHA-256 per file, the git commit that wrote it,
and the normalization table its `text` field went through (D64). The checksums are recomputed
here; if one disagrees, stop, because the numbers below would describe a different file from the
one the manifest documents.

`training` holds the `train` and `val` splits; `gold` holds `test`. A clip's `text` is its
current label written through the D64 table, exactly as the harness would score it.

In [ ]:
EXPORT_KINDS = ("training", "gold", "analytics")
manifests, rows_by_kind = {}, {}
for kind in EXPORT_KINDS:
    root = DATA_ROOT / kind
    if (root / "manifest.json").is_file() and (root / f"{kind}.jsonl").is_file():
        manifests[kind] = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
        with (root / f"{kind}.jsonl").open(encoding="utf-8") as handle:
            rows_by_kind[kind] = [json.loads(line) for line in handle if line.strip()]
if "training" not in manifests:
    raise SystemExit(f"no training export under {DATA_ROOT}; see the recipe at the top")


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 22), b""):
            digest.update(block)
    return digest.hexdigest()


expected = {
    DATA_ROOT / kind / entry["name"]: entry["sha256"]
    for kind, manifest in manifests.items()
    for entry in manifest["files"]
}
present = [path for path in expected if path.is_file()]
t0 = time.perf_counter()
with cf.ThreadPoolExecutor(max_workers=min(16, os.cpu_count() or 4)) as pool:
    digests = dict(zip(present, pool.map(sha256_of, present), strict=True))
bad = [str(p.relative_to(DATA_ROOT)) for p in present if digests[p] != expected[p]]
missing = [str(p.relative_to(DATA_ROOT)) for p in expected if not p.is_file()]
print(
    f"sha256: {len(present)} files, {sum(p.stat().st_size for p in present) / 1e9:.2f} GB "
    f"in {time.perf_counter() - t0:.1f}s"
)
if missing:
    print(f"  not uploaded (fine if unused): {missing}")
if bad:
    raise SystemExit(f"checksum mismatch, the files are not the ones the manifest describes: {bad}")

provenance = pd.DataFrame(
    [
        {
            "export": kind,
            "rows": m["row_count"],
            "exported_at": m["exported_at"][:19],
            "git_commit": (m.get("git_commit") or "")[:10],
            "label_version": m["label_version"],
            "policy": m["policy_version"],
            "normalization": m["normalization_version"],
        }
        for kind, m in manifests.items()
    ]
).set_index("export")
provenance

In [ ]:
def rows_frame(kind):
    frame = pd.DataFrame(rows_by_kind.get(kind, []))
    if frame.empty:
        return frame
    frame["seconds"] = frame["end_time"] - frame["start_time"]
    frame["export"] = kind
    return frame


train_rows, gold_rows = rows_frame("training"), rows_frame("gold")
corpus = pd.concat([train_rows, gold_rows], ignore_index=True)

assert set(train_rows["split"]) <= {"train", "val"}, (
    "the training export should hold train/val only"
)
if not gold_rows.empty:
    assert set(gold_rows["split"]) == {"test"}
    assert (gold_rows["verification_tier"] == "verified").all(), "the gold export promises verified"

composition = (
    corpus.groupby(["split", "verification_tier", "disposition"])
    .agg(clips=("segment_id", "size"), hours=("seconds", lambda s: s.sum() / 3600))
    .round({"hours": 2})
)
display(composition)

screened_share = (train_rows["verification_tier"] == "screened").mean()
print(
    f"episodes: {corpus['episode_id'].nunique()}   "
    f"train+val {train_rows['seconds'].sum() / 3600:.2f} h   "
    f"gold {gold_rows['seconds'].sum() / 3600 if not gold_rows.empty else 0:.2f} h"
)
print(
    f"{screened_share:.0%} of the training export is screened -- "
    f"{'INCLUDED' if INCLUDE_SCREENED else 'EXCLUDED'} by INCLUDE_SCREENED"
)

## 4. Splits and leakage checks

`val` is chosen per episode at import (a hash against `dataset.val_fraction`), so a small corpus
can easily have no `val` episode at all. When it has fewer than `VAL_MIN_CLIPS`, a
deterministic slice of train clips is held out instead. That slice shares episodes, and so
speakers, with train, so it is fine for picking a checkpoint and says nothing about
generalisation. Gold is never used to pick anything.

The assertions check what can be checked from the rows: no clip appears in two splits, and no
gold clip overlaps a training clip in time within the same episode.

In [ ]:
TIERS = ("verified", "screened") if INCLUDE_SCREENED else ("verified",)


def stable_fraction(key):
    """A uniform number in [0, 1) that depends only on `key`: the same carve on every run."""
    return int(hashlib.sha1(key.encode("utf-8")).hexdigest()[:12], 16) / 16**12


usable = train_rows[train_rows["verification_tier"].isin(TIERS)]
train_df = usable[usable["split"] == "train"].copy()
val_df = usable[usable["split"] == "val"].copy()
VAL_SOURCE = "val split (held-out episodes)"
if len(val_df) < VAL_MIN_CLIPS:
    carve = train_df["segment_id"].map(stable_fraction) < VAL_FALLBACK_FRACTION
    val_df = pd.concat([val_df, train_df[carve]])
    train_df = train_df[~carve]
    VAL_SOURCE = (
        f"fallback: {carve.sum()} train clips held out by hash (same episodes as train, so "
        "optimistic; for checkpoint selection only)"
    )
gold_df = gold_rows.copy()

split_ids = {
    "train": set(train_df["segment_id"]),
    "val": set(val_df["segment_id"]),
    "gold": set(gold_df["segment_id"]),
}
for a, b in (("train", "val"), ("train", "gold"), ("val", "gold")):
    assert not split_ids[a] & split_ids[b], f"{a} and {b} share clips"

# Same episode, overlapping time: the model would have heard part of a gold clip.
overlaps = 0
for episode_id, gold_clips in gold_df.groupby("episode_id"):
    fit = pd.concat([train_df, val_df])
    fit = fit[fit["episode_id"] == episode_id]
    for g in gold_clips.itertuples():
        overlaps += int(
            (
                (fit["start_time"] < g.end_time - 0.01) & (fit["end_time"] > g.start_time + 0.01)
            ).sum()
        )
assert overlaps == 0, f"{overlaps} train/val clips overlap a gold clip in time"

split_summary = pd.DataFrame(
    [
        {
            "split": name,
            "clips": len(df),
            "hours": round(df["seconds"].sum() / 3600, 3),
            "episodes": df["episode_id"].nunique(),
            "verified": int((df["verification_tier"] == "verified").sum()),
            "screened": int((df["verification_tier"] == "screened").sum()),
        }
        for name, df in (("train", train_df), ("val", val_df), ("gold", gold_df))
    ]
).set_index("split")
display(split_summary)
print(f"val source: {VAL_SOURCE}")
if not gold_df.empty:
    spans = gold_df["episode_spans_pots"].mean()
    print(
        f"gold clips whose episode also feeds train: {spans:.0%} "
        "(reported separately in section 11)"
    )

## 5. Text targets and the character vocabulary

The model is trained to emit what the harness's WER actually compares: the label with
punctuation and the danda removed, zero-width joiners dropped, Latin lowercased, one space
between words. Nothing the metric can see is thrown away, and the classes the metric ignores
(punctuation, case) are not spent on the output layer. The check below confirms that the
target's words are exactly `fold.py`'s raw WER tokens.

The vocabulary is every character in the *train* targets. Characters that only occur in val or
gold map to `<unk>` and are counted, not silently added. Characters are Unicode code points, so
a Devanagari vowel sign is its own symbol, which is how CTC is normally applied to Devanagari.

**CTC needs enough frames.** A clip is only learnable if its frame count is at least its label
length plus one blank between every repeated character. The encoder emits 50 frames a second
without its adapter and 25 with it. Section 6 measures both against this corpus: on the pilot
labels the fastest clip needs 0.50× its frames at 50/s and 1.00× (no slack at all) at 25/s,
which is why the adapter stays off.

In [ ]:
FOLD_AVAILABLE = (HARNESS_SRC / "backend" / "app" / "services" / "fold.py").is_file()
if FOLD_AVAILABLE:
    sys.path.insert(0, str(HARNESS_SRC / "backend"))
    # Private, and used only by the check below that the targets tokenize as fold does.
    from app.services.fold import _raw_tokens as fold_raw_tokens
    from app.services.fold import fold_version, word_errors

    FOLD_VERSION = fold_version()
    print(f"scoring with the harness fold: {FOLD_VERSION}")
else:
    FOLD_VERSION = None
    print(f"WARNING: {HARNESS_SRC} has no fold.py -- plain WER only, not comparable with EDA")

_JOINERS = str.maketrans("", "", "‌‍")
#: fold.py's `_NON_WORD`: Latin, digits, apostrophe, Devanagari letters and digits. The danda
#: (U+0964/U+0965) is outside both Devanagari ranges on purpose.
_NON_WORD = re.compile(r"[^A-Za-z0-9'ऀ-ॣ०-ॿ]+")


def ctc_target(text):
    """The label as the model should emit it: fold's raw WER tokens, joined by single spaces."""
    text = unicodedata.normalize("NFC", text or "").translate(_JOINERS)
    return " ".join(_NON_WORD.sub(" ", text).split()).lower()


for df in (train_df, val_df, gold_df):
    if not df.empty:
        df["target"] = df["text"].map(ctc_target)

if FOLD_AVAILABLE:
    mismatched = [
        t
        for t in pd.concat([train_df, val_df, gold_df])["text"]
        if ctc_target(t).split() != [w.lower() for w in fold_raw_tokens(t.translate(_JOINERS))]
    ]
    assert not mismatched, f"target and fold tokens disagree, e.g. {mismatched[0]!r}"
    print("targets == fold raw tokens on every clip")

empty = train_df["target"].str.len() == 0
if empty.any():
    print(f"dropping {empty.sum()} train clips whose label has no words")
    train_df = train_df[~empty]

In [ ]:
PAD, UNK, WORD = "<pad>", "<unk>", "|"
chars = sorted({c for t in train_df["target"] for c in t} - {" "})
vocab = {PAD: 0, UNK: 1, WORD: 2, **{c: i + 3 for i, c in enumerate(chars)}}

PROCESSOR_DIR = OUTPUT_DIR / "processor"
PROCESSOR_DIR.mkdir(parents=True, exist_ok=True)
(PROCESSOR_DIR / "vocab.json").write_text(json.dumps(vocab, ensure_ascii=False, indent=1), "utf-8")
# No <s>/</s>: CTC has no use for them, and adding them grows the output layer by two classes
# that are never trained.
tokenizer = Wav2Vec2CTCTokenizer(
    str(PROCESSOR_DIR / "vocab.json"),
    unk_token=UNK,
    pad_token=PAD,
    word_delimiter_token=WORD,
    bos_token=None,
    eos_token=None,
    do_lower_case=False,
)
feature_extractor = SeamlessM4TFeatureExtractor.from_pretrained(MODEL_ID)
processor = Wav2Vec2BertProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)
BLANK_ID = tokenizer.pad_token_id  # HF's CTC convention: the pad token is the blank
assert len(tokenizer) == len(vocab) and BLANK_ID == 0

for df in (train_df, val_df, gold_df):
    if not df.empty:
        df["label_ids"] = [np.asarray(tokenizer(t).input_ids, dtype=np.int32) for t in df["target"]]

roundtrip = [
    t
    for t, ids in zip(train_df["target"], train_df["label_ids"], strict=True)
    if tokenizer.decode(ids, group_tokens=False) != t
]
assert not roundtrip, f"tokenizer does not round-trip {roundtrip[0]!r}"

latin = sum(c.isascii() and c.isalpha() for c in chars)
devanagari = sum("ऀ" <= c <= "ॿ" for c in chars)
print(
    f"vocabulary: {len(vocab)} symbols ({devanagari} Devanagari, {latin} Latin, "
    f"{len(chars) - latin - devanagari} other) + blank/unk/delimiter"
)
for name, df in (("val", val_df), ("gold", gold_df)):
    if not df.empty:
        total = sum(len(ids) for ids in df["label_ids"])
        unk = sum(int((ids == vocab[UNK]).sum()) for ids in df["label_ids"])
        print(
            f"{name}: {unk} of {total} label symbols are outside the train vocabulary "
            f"({unk / max(total, 1):.2%})"
        )

## 6. Audio → features

The export's `audio_path` names `clips/<segment_id>.flac`, but the export writes no `clips/`
directory; it writes each whole episode under `episodes/`. A clip is therefore recut from its
episode at `start_time`/`end_time` with the same rounding ingest used (`round(t * 16000)`).
Checked against the stored clips on this corpus, that reproduces them sample for sample; the only
difference is the 5 ms anti-click fade ingest puts on each end. If a `clips/` directory is
present (flat, or the object store's `clips/<episode>/` layout) it is used instead.

Features are w2v-BERT's own: 80 log-mel bins every 10 ms, two frames stacked into one 160-dim
vector every 20 ms, normalised per clip. They are computed once, in parallel, and kept in host
memory as float16 (14 h is about 0.8 GB).

In [ ]:
EXPORT_DIRS = [DATA_ROOT / kind for kind in EXPORT_KINDS if (DATA_ROOT / kind).is_dir()]
SAMPLE_RATE = feature_extractor.sampling_rate
FRAMES_PER_SECOND = SAMPLE_RATE / 160 / feature_extractor.stride  # 10 ms hop, then stacked
FEATURE_DIM = feature_extractor.feature_size * feature_extractor.stride
PAD_VALUE = float(feature_extractor.padding_value)


def locate_audio(row):
    """(file, start, end): a clip file if one was uploaded, else the episode and a time span."""
    for root in EXPORT_DIRS:
        for candidate in (
            root / row["audio_path"],
            root / "clips" / row["episode_id"] / f"{row['segment_id']}.flac",
        ):
            if candidate.is_file():
                return candidate, None, None
    for root in EXPORT_DIRS:
        episode = root / "episodes" / f"{row['episode_id']}.flac"
        if episode.is_file():
            return episode, row["start_time"], row["end_time"]
    raise FileNotFoundError(f"no audio for {row['segment_id']} (episode {row['episode_id']})")


@functools.cache
def audio_info(path):
    info = sf.info(str(path))
    if info.samplerate != SAMPLE_RATE or info.channels != 1:
        raise ValueError(
            f"{path.name}: {info.samplerate} Hz x{info.channels}, expected 16 kHz mono"
        )
    return info


def features_for(row):
    path, start, end = locate_audio(row)
    audio_info(path)
    if start is None:
        audio, _ = sf.read(str(path), dtype="float32")
    else:
        audio, _ = sf.read(
            str(path),
            start=round(start * SAMPLE_RATE),
            stop=round(end * SAMPLE_RATE),
            dtype="float32",
        )
    feats = feature_extractor(
        audio, sampling_rate=SAMPLE_RATE, return_attention_mask=False, return_tensors="np"
    )["input_features"][0]
    return row["segment_id"], feats.astype(np.float16)


all_rows = pd.concat([train_df, val_df, gold_df], ignore_index=True)
t0 = time.perf_counter()
# Threads, not processes: libsndfile and numpy's FFT release the GIL, and nothing is forked
# after the XLA runtime has started.
with cf.ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as pool:
    FEATURES = dict(pool.map(features_for, all_rows.to_dict("records")))
elapsed = time.perf_counter() - t0
audio_hours = all_rows["seconds"].sum() / 3600
print(
    f"features: {len(FEATURES)} clips, {audio_hours:.2f} h of audio in {elapsed:.1f}s "
    f"on {os.cpu_count()} threads, "
    f"{sum(f.nbytes for f in FEATURES.values()) / 1e9:.2f} GB in memory"
)
print(f"frame rate {FRAMES_PER_SECOND:g}/s, feature dim {FEATURE_DIM}, pad value {PAD_VALUE}")

In [ ]:
def ctc_frames_needed(ids):
    """Minimum CTC frames for a label: one per symbol, plus a blank between each repeated pair."""
    return len(ids) + int((ids[1:] == ids[:-1]).sum())


for df in (train_df, val_df, gold_df):
    if not df.empty:
        df["frames"] = df["segment_id"].map(lambda s: len(FEATURES[s]))
        df["need"] = df["label_ids"].map(ctc_frames_needed)

fit = pd.concat([train_df, val_df])
for fps in (FRAMES_PER_SECOND, FRAMES_PER_SECOND / 2):
    ratio = fit["need"] / (fit["frames"] * fps / FRAMES_PER_SECOND)
    print(
        f"at {fps:g} frames/s: worst clip needs {ratio.max():.2f}x its frames; "
        f"{int((ratio > 1).sum())} clips unlearnable"
    )


def keep(df, name):
    ok = (df["seconds"].between(MIN_SECONDS, MAX_SECONDS)) & (df["need"] <= df["frames"])
    if (~ok).any():
        print(
            f"{name}: dropping {(~ok).sum()} clips (outside {MIN_SECONDS}-{MAX_SECONDS} s "
            "or too short for their label)"
        )
    return df[ok].reset_index(drop=True)


train_df, val_df = keep(train_df, "train"), keep(val_df, "val")
if not gold_df.empty:
    # Gold is scored whatever its length; a clip too long for a bucket is scored on its own.
    gold_df = gold_df.reset_index(drop=True)

## 7. Static-shape batching

XLA compiles one program per distinct input shape, so batches come in a fixed set of shapes:
each clip goes into the smallest bucket that holds it, every batch in a bucket is padded to that
bucket's length, and the batch size is fixed per bucket so that each step sees about
`BATCH_AUDIO_SECONDS` of audio. A bucket's last batch is topped up with clips from the start of
its shuffled order, so nothing is dropped and no padding row ever has an empty attention mask
(which would produce NaN).

**SpecAugment is applied here, on the host,** rather than through the model's `mask_time_prob`.
HF's implementation writes the mask with boolean indexing, a data-dependent shape that XLA either
recompiles for or pulls back to the host, every step.

In [ ]:
def bucket_frames(seconds):
    return int(math.ceil(seconds * FRAMES_PER_SECOND / 8) * 8)  # multiple of 8 for the TPU tiles


BUCKETS = []
for seconds in BUCKET_SECONDS:
    per_device = max(1, round(BATCH_AUDIO_SECONDS / seconds / N_DEVICES))
    BUCKETS.append(
        {"seconds": seconds, "frames": bucket_frames(seconds), "batch": per_device * N_DEVICES}
    )
BUCKET_LIMITS = np.array([b["frames"] for b in BUCKETS])


def bucket_of(n_frames):
    index = int(np.searchsorted(BUCKET_LIMITS, n_frames))
    return index if index < len(BUCKETS) else None


class ClipSet:
    """A split's features, label ids and bucket assignment, indexed by position."""

    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.feats = [FEATURES[s] for s in self.df["segment_id"]]
        self.labels = list(self.df["label_ids"])
        self.buckets = [bucket_of(len(f)) for f in self.feats]
        self.members = [
            [i for i, b in enumerate(self.buckets) if b == k] for k in range(len(BUCKETS))
        ]

    def __len__(self):
        return len(self.df)


def spec_augment(feats, rng):
    """Frequency masks on the same mel bins in both stacked halves, then short time masks."""
    feats = feats.copy()
    mels = feature_extractor.feature_size
    for _ in range(SPEC_AUGMENT["freq_masks"]):
        width = rng.integers(0, SPEC_AUGMENT["freq_width"] + 1)
        start = rng.integers(0, mels - width + 1)
        feats[:, start : start + width] = 0.0
        feats[:, mels + start : mels + start + width] = 0.0
    n_frames = len(feats)
    n_masks = int(
        n_frames * SPEC_AUGMENT["time_mask_rate"] / SPEC_AUGMENT["time_width"] + rng.random()
    )
    for _ in range(n_masks):
        width = rng.integers(1, SPEC_AUGMENT["time_width"] + 1)
        start = rng.integers(0, max(1, n_frames - width))
        feats[start : start + width] = 0.0
    return feats


def collate(clips, indices, bucket, rng=None):
    """Pad to the bucket's fixed (batch, frames). `rng` switches SpecAugment on."""
    spec = BUCKETS[bucket]
    batch, frames = spec["batch"], spec["frames"]
    real = len(indices)
    indices = np.resize(np.asarray(indices), batch)  # top up with the batch's own clips
    features = np.full((batch, frames, FEATURE_DIM), PAD_VALUE, dtype=np.float32)
    mask = np.zeros((batch, frames), dtype=np.int64)
    lengths = np.zeros(batch, dtype=np.int64)
    for row, index in enumerate(indices):
        f = clips.feats[index].astype(np.float32)
        if rng is not None:
            f = spec_augment(f, rng)
        features[row, : len(f)], mask[row, : len(f)], lengths[row] = f, 1, len(f)
    labels = [clips.labels[i] for i in indices]
    padded = np.zeros((batch, max(len(ids) for ids in labels)), dtype=np.int64)
    for row, ids in enumerate(labels):
        padded[row, : len(ids)] = ids
    return {
        "features": torch.from_numpy(features),
        "mask": torch.from_numpy(mask),
        "input_lengths": torch.from_numpy(lengths),
        "labels": torch.from_numpy(padded),
        "label_lengths": torch.tensor([len(ids) for ids in labels], dtype=torch.int64),
        "indices": indices[:real],
    }


def training_batches(clips, rng):
    """One epoch: every clip once (a few twice, topping up), buckets interleaved at random."""
    plan = []
    for bucket, members in enumerate(clips.members):
        if not members:
            continue
        order = rng.permutation(members)
        size = BUCKETS[bucket]["batch"]
        order = np.resize(order, math.ceil(len(order) / size) * size)
        plan += [(bucket, order[i : i + size]) for i in range(0, len(order), size)]
    rng.shuffle(plan)
    return plan


def evaluation_batches(clips):
    for bucket, members in enumerate(clips.members):
        size = BUCKETS[bucket]["batch"]
        for i in range(0, len(members), size):
            yield bucket, members[i : i + size]


train_set, val_set = ClipSet(train_df), ClipSet(val_df)
gold_set = ClipSet(gold_df) if not gold_df.empty else None
STEPS_PER_EPOCH = len(training_batches(train_set, np.random.default_rng(0)))
TOTAL_STEPS = STEPS_PER_EPOCH * EPOCHS

plan = pd.DataFrame(BUCKETS)
plan["train clips"] = [len(m) for m in train_set.members]
plan["val clips"] = [len(m) for m in val_set.members]
plan["gold clips"] = [len(m) for m in gold_set.members] if gold_set else 0
display(plan.set_index("seconds"))
real = sum(len(f) for f in train_set.feats)
padded = sum(BUCKETS[b]["frames"] for b in train_set.buckets)
print(
    f"padding waste {1 - real / padded:.0%}   {STEPS_PER_EPOCH} steps/epoch x {EPOCHS} epochs "
    f"= {TOTAL_STEPS} steps   graphs to compile: {3 * len(BUCKETS)}"
)
overlong = sum(b is None for b in gold_set.buckets) if gold_set else 0
if overlong:
    print(f"{overlong} gold clips exceed the largest bucket and are transcribed one at a time")

## 8. Model

Four changes to the pretrained configuration, each forced by the TPU or by this corpus:

| setting | pretrained | here | why |
|---|---|---|---|
| `layerdrop` | 0.1 | 0 | skipping random layers changes the graph every step, so XLA would recompile every step |
| `mask_time_prob` | 0.05 | 0 | replaced by host-side SpecAugment (section 7) |
| `add_adapter` | off | off | the adapter halves the frame rate to 25/s; section 6 prints how close the fastest speech here comes to needing every one of those frames |
| `nn.GLU` | GLU | `a * sigmoid(b)` | same function; its backward has an XLA lowering, `glu_backward` does not |

The CTC head is new and randomly initialised; everything else starts from the pretrained
encoder. Gradient checkpointing recomputes each conformer layer in the backward pass instead of
storing its activations; on the TPU it goes through `torch_xla`'s version, which the compiler
cannot optimise away.

In [ ]:
class XlaGLU(torch.nn.Module):
    """`nn.GLU` without `aten::glu_backward`, which PyTorch/XLA runs on the CPU."""

    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        a, b = x.chunk(2, dim=self.dim)
        return a * torch.sigmoid(b)


def replace_glu(module):
    for parent in module.modules():
        for name, child in list(parent.named_children()):
            if isinstance(child, torch.nn.GLU):
                setattr(parent, name, XlaGLU(child.dim))


class CTCLogits(torch.nn.Module):
    """The model as a function of (features, mask) -> logits: a plain tensor, which is what
    the sharding annotations and FSDPv2 expect, and all the training loop needs."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, features, mask):
        return self.model(input_features=features, attention_mask=mask).logits


overrides = {
    "vocab_size": len(tokenizer),
    "pad_token_id": BLANK_ID,
    "layerdrop": 0.0,
    "mask_time_prob": 0.0,
    "mask_feature_prob": 0.0,
    "add_adapter": False,
    # Only used if someone calls the model with `labels=`; the loop below computes its own loss.
    "ctc_loss_reduction": "mean",
    "ctc_zero_infinity": True,
}
if SMOKE_TEST:
    config = Wav2Vec2BertConfig(
        hidden_size=64,
        num_hidden_layers=2,
        num_attention_heads=4,
        intermediate_size=128,
        output_hidden_size=64,
        **overrides,
    )
    model = Wav2Vec2BertForCTC(config)
else:
    model = Wav2Vec2BertForCTC.from_pretrained(MODEL_ID, **overrides)
replace_glu(model)
print(
    f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f} M parameters, "
    f"output layer {model.lm_head.out_features} classes"
)

if USE_XLA:
    from torch_xla.distributed.fsdp.utils import checkpoint_module

    if PARALLELISM == "fsdp":
        from torch_xla.distributed.fsdp.wrap import transformer_auto_wrap_policy
        from torch_xla.experimental.spmd_fully_sharded_data_parallel import (
            SpmdFullyShardedDataParallel as FSDPv2,
        )

        def _wrap_layer(layer, *args, **kwargs):
            return FSDPv2(
                checkpoint_module(layer) if GRADIENT_CHECKPOINTING else layer, *args, **kwargs
            )

        net = FSDPv2(
            CTCLogits(model),
            mesh=MESH,
            auto_wrap_policy=functools.partial(
                transformer_auto_wrap_policy, transformer_layer_cls={Wav2Vec2BertEncoderLayer}
            ),
            auto_wrapper_callable=_wrap_layer,
        )
    else:
        if GRADIENT_CHECKPOINTING:
            for layer in model.wav2vec2_bert.encoder.layers:
                checkpoint_module(layer)
        net = CTCLogits(model).to(DEVICE)  # parameters replicated; the batch is sharded
else:
    if GRADIENT_CHECKPOINTING:
        model.gradient_checkpointing_enable()
    net = CTCLogits(model).to(DEVICE)

optimizer = torch.optim.AdamW(net.parameters(), lr=PEAK_LR, weight_decay=WEIGHT_DECAY)
WARMUP_STEPS = max(1, int(WARMUP_FRACTION * TOTAL_STEPS))


def lr_lambda(step):
    """Linear warmup, then linear decay to zero."""
    if step < WARMUP_STEPS:
        return (step + 1) / WARMUP_STEPS
    return max(0.0, (TOTAL_STEPS - step) / max(1, TOTAL_STEPS - WARMUP_STEPS))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
print(
    f"{PARALLELISM if USE_XLA else 'single device'}, checkpointing {GRADIENT_CHECKPOINTING}, "
    f"warmup {WARMUP_STEPS} of {TOTAL_STEPS} steps, peak lr {PEAK_LR}"
)

## 9. Loss, decoding and scoring

**Loss.** Per-clip CTC divided by label length, averaged over the clips in the batch (PyTorch's
`reduction="mean"`), computed on the host for the reason in section 2. Top-up duplicates carry
no weight.

**Decoding** is greedy: argmax per frame on the device, then collapse repeats and drop blanks.

**Scoring** uses the harness's own `word_errors` twice. **Raw** WER is a plain WER over
punctuation-stripped words. **Folded** WER forgives what `fold.py` forgives (spelling variants, a
word written in the other script, a missing or extra space) and is the number to compare with
`EDA`. CER is over the targets. Intervals are percentile bootstraps over clips; with this few
episodes an episode-clustered interval cannot be computed, and a clip-level interval is too
narrow because clips from one episode are not independent.

In [ ]:
def autocast():
    if AUTOCAST_DTYPE is None:
        return torch.autocast("cpu", enabled=False)
    return torch.autocast("xla" if USE_XLA else DEVICE.type, dtype=AUTOCAST_DTYPE)


def to_device(batch):
    features, mask = batch["features"].to(DEVICE), batch["mask"].to(DEVICE)
    if USE_XLA:
        xs.mark_sharding(features, MESH, ("fsdp", None, None))
        xs.mark_sharding(mask, MESH, ("fsdp", None))
    return features, mask


def ctc_loss(log_probs, batch):
    """CTC on the host: PyTorch/XLA has no lowering for it. `.to("cpu")` is differentiable, so
    the gradient of the logits travels back to the device through the same transfer."""
    if USE_XLA:
        log_probs = log_probs.to("cpu")
    device = log_probs.device
    per_clip = F.ctc_loss(
        log_probs.transpose(0, 1),
        batch["labels"].to(device),
        batch["input_lengths"].to(device),
        batch["label_lengths"].to(device),
        blank=BLANK_ID,
        reduction="none",
        zero_infinity=True,
    )
    weights = torch.zeros_like(per_clip)
    weights[: len(batch["indices"])] = 1.0
    return (
        per_clip / batch["label_lengths"].to(device).clamp(min=1) * weights
    ).sum() / weights.sum()


def train_step(batch):
    features, mask = to_device(batch)
    with autocast():
        logits = net(features, mask)
    loss = ctc_loss(logits.float().log_softmax(-1), batch)
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(net.parameters(), MAX_GRAD_NORM)
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)
    sync()
    return float(loss.detach()), grad_norm


def decode(ids, length):
    return " ".join(tokenizer.decode(ids[:length]).split())


@torch.no_grad()
def transcribe(clips):
    """Greedy CTC transcripts for every clip in a ClipSet, keyed by segment id."""
    net.eval()
    out = {}
    for bucket, members in evaluation_batches(clips):
        batch = collate(clips, members, bucket)
        features, mask = to_device(batch)
        with autocast():
            ids = net(features, mask).argmax(-1)
        ids = ids.cpu().numpy()
        for row, index in enumerate(batch["indices"]):
            out[clips.df.at[index, "segment_id"]] = decode(
                ids[row], int(batch["input_lengths"][row])
            )
    # Longer than the largest bucket (gold only): one at a time, at its own length. Rare, and
    # each distinct length compiles once.
    for index in (i for i, b in enumerate(clips.buckets) if b is None):
        f = torch.from_numpy(clips.feats[index].astype(np.float32))[None].to(DEVICE)
        m = torch.ones(f.shape[:2], dtype=torch.int64, device=DEVICE)
        with autocast():
            ids = CTCLogits(model)(f, m).argmax(-1).cpu().numpy()
        out[clips.df.at[index, "segment_id"]] = decode(ids[0], f.shape[1])
    net.train()
    return out


def clip_scores(df, hyps):
    """Per-clip error counts under both metrics."""
    records = []
    for row in df.itertuples():
        hyp = hyps[row.segment_id]
        rec = {
            "segment_id": row.segment_id,
            "episode_id": row.episode_id,
            "ref": row.text,
            "hyp": hyp,
            "chars": len(row.target),
            "char_errors": Levenshtein.distance(row.target, hyp),
        }
        if FOLD_AVAILABLE:
            raw, folded = word_errors(row.text, hyp, folded=False), word_errors(row.text, hyp)
            rec.update(
                words=raw.ref_words,
                raw_errors=raw.errors,
                folded_words=folded.ref_words,
                folded_errors=folded.errors,
            )
        else:
            ref_words = row.target.split()
            rec.update(
                words=len(ref_words), raw_errors=Levenshtein.distance(ref_words, hyp.split())
            )
        records.append(rec)
    return pd.DataFrame(records)


def pooled(scores, errors, words, n_boot=2000, seed=0):
    """Pooled rate and a clip-level 95% bootstrap interval."""
    e, w = scores[errors].to_numpy(float), scores[words].to_numpy(float)
    rate = e.sum() / max(w.sum(), 1)
    idx = np.random.default_rng(seed).integers(0, len(e), (n_boot, len(e)))
    boot = e[idx].sum(1) / np.maximum(w[idx].sum(1), 1)
    return rate, *np.percentile(boot, [2.5, 97.5])


def summarise(scores):
    out = {
        "clips": len(scores),
        "cer": pooled(scores, "char_errors", "chars")[0],
        "raw_wer": pooled(scores, "raw_errors", "words")[0],
    }
    if FOLD_AVAILABLE:
        out["folded_wer"] = pooled(scores, "folded_errors", "folded_words")[0]
    return out


SELECT_ON = "folded_wer" if FOLD_AVAILABLE else "raw_wer"

## 10. Training

The first step of each bucket compiles its graph, so the first epoch is slow; the compile
count in the log should stop rising once every bucket has been seen. Validation runs after every
epoch, and the checkpoint with the best validation WER is written to `best/` in a
`from_pretrained`-loadable form.

In [ ]:
BEST_DIR = OUTPUT_DIR / "best"


def host_state_dict():
    """Weights on the host under their plain HF names (FSDPv2 inserts `_orig_module.`)."""
    return {
        k.replace("_orig_module.", ""): v.detach().to("cpu").clone()
        for k, v in model.state_dict().items()
    }


def load_host_state_dict(state):
    """Copy host weights back into the live (possibly sharded) parameters, in place."""
    with torch.no_grad():
        for name, tensor in list(model.named_parameters()) + list(model.named_buffers()):
            clean = name.replace("_orig_module.", "")
            if clean in state:
                tensor.copy_(state[clean].to(tensor.device, tensor.dtype))
    sync()


def save_checkpoint(state, directory):
    model.save_pretrained(directory, state_dict=state, safe_serialization=True)
    processor.save_pretrained(directory)


history, best = [], {SELECT_ON: math.inf, "step": None, "state": None}
rng = np.random.default_rng(SEED)
step, t_start, audio_seen = 0, time.perf_counter(), 0.0
deadline = t_start + MAX_TRAIN_HOURS * 3600
net.train()
for epoch in range(1, EPOCHS + 1):
    epoch_loss, epoch_steps, t_epoch = 0.0, 0, time.perf_counter()
    for bucket, indices in training_batches(train_set, rng):
        batch = collate(train_set, indices, bucket, rng)
        loss, grad_norm = train_step(batch)
        step += 1
        epoch_loss, epoch_steps = epoch_loss + loss, epoch_steps + 1
        audio_seen += sum(len(train_set.feats[i]) for i in batch["indices"]) / FRAMES_PER_SECOND
        if step % LOG_EVERY_STEPS == 0 or step == 1:
            rate = audio_seen / 3600 / ((time.perf_counter() - t_start) / 60)
            print(
                f"step {step:5d}/{TOTAL_STEPS}  epoch {epoch}  loss {loss:7.4f}  "
                f"grad {float(grad_norm):6.2f}  lr {scheduler.get_last_lr()[0]:.2e}  "
                f"{rate:5.2f} audio-h/min  compiles {compile_count()}"
            )
        if time.perf_counter() > deadline:
            break

    hyps = transcribe(val_set)
    metrics = summarise(clip_scores(val_set.df, hyps))
    history.append(
        {
            "epoch": epoch,
            "step": step,
            "train_loss": epoch_loss / max(epoch_steps, 1),
            **{f"val_{k}": v for k, v in metrics.items() if k != "clips"},
            "minutes": (time.perf_counter() - t_start) / 60,
        }
    )
    improved = metrics[SELECT_ON] < best[SELECT_ON]
    if improved:
        best.update(
            {
                SELECT_ON: metrics[SELECT_ON],
                "step": step,
                "epoch": epoch,
                "state": host_state_dict(),
            }
        )
        save_checkpoint(best["state"], BEST_DIR)
    example = val_set.df.iloc[0]
    print(
        f"--- epoch {epoch}: train loss {history[-1]['train_loss']:.4f}  val "
        + "  ".join(f"{k} {v:.3f}" for k, v in metrics.items() if k != "clips")
        + f"  ({time.perf_counter() - t_epoch:.0f}s){'  * best' if improved else ''}"
    )
    print(f"    ref: {example['target'][:110]}\n    hyp: {hyps[example['segment_id']][:110]}")
    if time.perf_counter() > deadline:
        print(f"stopping: MAX_TRAIN_HOURS={MAX_TRAIN_HOURS} reached")
        break

TRAIN_MINUTES = (time.perf_counter() - t_start) / 60
print(
    f"trained {step} steps in {TRAIN_MINUTES:.1f} min; best {SELECT_ON} {best[SELECT_ON]:.3f} "
    f"at epoch {best.get('epoch')}; graphs compiled {compile_count()}"
)

In [ ]:
curve = pd.DataFrame(history)
curve.to_csv(OUTPUT_DIR / "training_curve.csv", index=False)
fig, (ax_loss, ax_wer) = plt.subplots(1, 2, figsize=(11, 3.6))
ax_loss.plot(curve["epoch"], curve["train_loss"], marker="o", ms=3)
ax_loss.set(xlabel="epoch", ylabel="CTC loss / label symbol", title="train loss")
for column in [c for c in curve if c.startswith("val_") and c != "val_cer"] + ["val_cer"]:
    ax_wer.plot(curve["epoch"], curve[column], marker="o", ms=3, label=column[4:])
ax_wer.axvline(best.get("epoch") or 0, color="grey", lw=0.8, ls="--")
ax_wer.set(
    xlabel="epoch",
    ylabel="error rate",
    title=f"validation ({len(val_set)} clips)",
    ylim=(0, min(1.2, max(1.0, curve.filter(like="val_").to_numpy().max()))),
)
ax_wer.legend()
fig.savefig(OUTPUT_DIR / "training_curve.png", dpi=120)
plt.show()

## 11. Gold evaluation

The best-on-validation weights, scored once on every gold clip. The split by
`episode_spans_pots` separates gold clips whose episode also contributed training clips, which
share speaker, room and topic with train, from those whose episode is entirely gold. The second
group is closer to a held-out number, but only at show level: the `EDA` voice prints found the same
voice across different shows.

In [ ]:
if gold_set is None:
    raise SystemExit("no gold export uploaded -- nothing to evaluate")

load_host_state_dict(best["state"])
gold_hyps = transcribe(gold_set)
gold_scores = clip_scores(gold_df, gold_hyps).merge(
    gold_df[["segment_id", "episode_spans_pots", "seconds"]], on="segment_id"
)


def report(scores):
    row = {"clips": len(scores), "hours": scores["seconds"].sum() / 3600}
    metrics = [("CER", "char_errors", "chars"), ("raw WER", "raw_errors", "words")]
    if FOLD_AVAILABLE:
        metrics.append(("folded WER", "folded_errors", "folded_words"))
    for label, errors, words in metrics:
        rate, lo, hi = pooled(scores, errors, words)
        row[label] = f"{rate:.1%} [{lo:.1%}, {hi:.1%}]"
    return row


gold_table = pd.DataFrame(
    {
        "all gold": report(gold_scores),
        **{
            f"episode also in train: {flag}": report(group)
            for flag, group in gold_scores.groupby("episode_spans_pots")
        },
    }
).T
display(gold_table)
by_episode = gold_scores.groupby("episode_id").apply(
    lambda g: pd.Series(report(g)), include_groups=False
)
display(by_episode)

In [ ]:
worst_key = "folded_errors" if FOLD_AVAILABLE else "raw_errors"
worst_words = "folded_words" if FOLD_AVAILABLE else "words"
gold_scores["wer"] = gold_scores[worst_key] / gold_scores[worst_words].clip(lower=1)
print("Worst gold clips (reference as labeled, then the model):\n")
for row in gold_scores.sort_values("wer", ascending=False).head(8).itertuples():
    print(f"{row.segment_id}  WER {row.wer:.0%}")
    print(f"  ref: {row.ref}\n  hyp: {row.hyp}\n")

## 12. The same gold clips, transcribed by the harness's recognisers

When the `analytics` export is uploaded it carries every recogniser's hypothesis for each gold
clip, so they can be scored with the same metric on the same clips. The fused hypothesis is
left out: it seeded these labels (D74), so on an accepted-unchanged clip its WER is zero by
construction. The three recognisers are not the seed, but the fuser built the seed from them,
so their numbers are probably flattered by anchoring as well. A fine-tuned model that is close
to them is doing better than this table makes it look.

In [ ]:
baseline_table = None
if "analytics" in rows_by_kind and FOLD_AVAILABLE:
    gold_ids = set(gold_df["segment_id"])
    refs = dict(zip(gold_df["segment_id"], gold_df["text"], strict=True))
    per_system = {}
    for record in rows_by_kind["analytics"]:
        if record["segment_id"] not in gold_ids:
            continue
        for hyp in record["hypotheses"]:
            if hyp["system_id"].startswith("fusion"):
                continue
            raw = word_errors(refs[record["segment_id"]], hyp["text"], folded=False)
            folded = word_errors(refs[record["segment_id"]], hyp["text"])
            per_system.setdefault(hyp["system_id"], []).append(
                (raw.errors, raw.ref_words, folded.errors, folded.ref_words)
            )
    ours = gold_scores[["raw_errors", "words", "folded_errors", "folded_words"]].to_numpy()
    per_system["fine-tuned (this run)"] = [tuple(r) for r in ours]
    baseline_table = (
        pd.DataFrame(
            [
                {
                    "system": name,
                    "clips": len(v),
                    "raw WER": sum(r[0] for r in v) / max(sum(r[1] for r in v), 1),
                    "folded WER": sum(r[2] for r in v) / max(sum(r[3] for r in v), 1),
                }
                for name, v in per_system.items()
            ]
        )
        .set_index("system")
        .sort_values("folded WER")
    )
    display(baseline_table.style.format({"raw WER": "{:.1%}", "folded WER": "{:.1%}"}))
else:
    print("skipped: needs the analytics export and the harness fold (see the recipe at the top)")

## 13. Artifacts

`best/` is a standard Hugging Face checkpoint. It is reloaded here on the CPU, from disk: every
tensor must equal the best-on-validation weights exactly, with no missing or unexpected key,
and a few gold clips are transcribed through the saved processor as a user of the checkpoint
would, then compared with what the TPU produced in bf16.

In [ ]:
gold_scores.drop(columns=["wer"]).to_json(
    OUTPUT_DIR / "gold_predictions.jsonl", orient="records", lines=True, force_ascii=False
)

reloaded, loading = Wav2Vec2BertForCTC.from_pretrained(BEST_DIR, output_loading_info=True)
reloaded.eval()
problems = {k: v for k, v in loading.items() if v}
assert not problems, f"checkpoint does not load cleanly: {problems}"
reloaded_state = reloaded.state_dict()
assert set(reloaded_state) == set(best["state"]), "saved and trained parameter names differ"
differ = [k for k, v in best["state"].items() if not torch.equal(reloaded_state[k], v)]
assert not differ, f"{len(differ)} saved tensors differ from the best weights, e.g. {differ[0]}"
print(f"best/ holds exactly the best-on-validation weights ({len(reloaded_state)} tensors)")
reloaded_processor = Wav2Vec2BertProcessor.from_pretrained(BEST_DIR)
sample = gold_df.head(4)
agree = []
for row in sample.itertuples():
    path, start, end = locate_audio(row._asdict())
    audio, _ = sf.read(
        str(path),
        start=None if start is None else round(start * SAMPLE_RATE),
        stop=None if end is None else round(end * SAMPLE_RATE),
        dtype="float32",
    )
    inputs = reloaded_processor(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt")
    with torch.no_grad():
        ids = reloaded(**inputs).logits.argmax(-1)
    text = " ".join(reloaded_processor.batch_decode(ids)[0].split())
    agree.append(text == gold_hyps[row.segment_id])
    print(f"cpu fp32: {text[:100]}\ntpu bf16: {gold_hyps[row.segment_id][:100]}\n")
print(
    f"reloaded checkpoint matches the in-memory transcript on {sum(agree)}/{len(agree)} clips "
    "(bf16 against fp32 can differ by a character or two)"
)

In [ ]:
def as_float(value):
    return (
        None if value is None or (isinstance(value, float) and math.isnan(value)) else float(value)
    )


summary = {
    "model": "tiny-random (SMOKE_TEST)" if SMOKE_TEST else MODEL_ID,
    "versions": VERSIONS,
    "accelerator": ACCELERATOR,
    "exports": {
        kind: {
            k: m.get(k)
            for k in (
                "exported_at",
                "git_commit",
                "label_version",
                "policy_version",
                "normalization_version",
                "row_count",
            )
        }
        | {"sha256": next(e["sha256"] for e in m["files"] if e["name"].endswith(".jsonl"))}
        for kind, m in manifests.items()
    },
    "fold_version": FOLD_VERSION,
    "data": {
        "include_screened": INCLUDE_SCREENED,
        "val_source": VAL_SOURCE,
        **{
            name: {"clips": len(s.df), "hours": round(float(s.df["seconds"].sum()) / 3600, 3)}
            for name, s in (("train", train_set), ("val", val_set), ("gold", gold_set))
        },
    },
    "vocab_size": len(tokenizer),
    "config": {
        "epochs": EPOCHS,
        "peak_lr": PEAK_LR,
        "warmup_steps": WARMUP_STEPS,
        "steps": step,
        "buckets": BUCKETS,
        "batch_audio_seconds": BATCH_AUDIO_SECONDS,
        "spec_augment": SPEC_AUGMENT,
        "parallelism": PARALLELISM,
        "gradient_checkpointing": GRADIENT_CHECKPOINTING,
        "seed": SEED,
    },
    "training_minutes": round(TRAIN_MINUTES, 1),
    "graphs_compiled": compile_count(),
    "best": {
        "epoch": best.get("epoch"),
        "step": best["step"],
        SELECT_ON: as_float(best[SELECT_ON]),
    },
    "gold": {k: as_float(v) if k != "clips" else v for k, v in summarise(gold_scores).items()},
    "baselines_on_gold": None
    if baseline_table is None
    else {
        k: {m: float(v) for m, v in row.items() if m != "clips"}
        for k, row in baseline_table.iterrows()
    },
}
(OUTPUT_DIR / "training_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)

if PUSH_TO_HUB_REPO and not SMOKE_TEST:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("HF_TOKEN")
    reloaded.push_to_hub(PUSH_TO_HUB_REPO, token=token, private=True)
    reloaded_processor.push_to_hub(PUSH_TO_HUB_REPO, token=token, private=True)

for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size / 1e6:9.1f} MB  {path.relative_to(OUTPUT_DIR)}")

## 14. Reading the pilot, and what comes next

**On an hour of labels, expect a high WER.** Several hundred CTC steps is enough for the model to
start producing recognisable words, not enough for it to be good. The pilot is working if:
the loss falls steadily; validation WER falls with it and then flattens; the compile count stops
rising after the first epoch; the reloaded checkpoint agrees with the TPU; and the gold examples
look like the audio with spelling mistakes rather than like noise.

**When the corpus is bigger**, in rough order of value:

1. **Re-run as is** with `EPOCHS` around 10-15. Nothing else in the notebook depends on corpus size.
2. **An n-gram LM** from the label text (KenLM), decoded with `pyctcdecode`. This is where CTC's
   English spelling improves, and it needs no retraining.
3. **Decide on screened rows with a measurement.** Train once with `INCLUDE_SCREENED = False`
   and compare gold WER. If the screened rows do not help, the 80% auto-approve target is also
   worth less than it looks.
4. **A speaker-disjoint gold slice.** Until one exists, no number from this notebook says how the
   model handles a voice it has never heard.